# 04 — The map: the Transformer architecture, and what we will keep of it

**Chapter 2** · suggested time: 20 min · no training · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/en/chapter_2/04_transformer_architecture.ipynb)

![Roadmap](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_2/figures/00_roadmap.png)

Before diving into the parts (attention, FeedForward, residuals), let's look at **the complete blueprint**. This notebook is deliberately light on code: its goal is that, at any point in what follows, everyone knows **where the part we are building fits**.

**By the end of this notebook you will be able to:**
1. read the diagram of the original Transformer (encoder + decoder);
2. explain the difference between **encoder**, **decoder** and **cross-attention**;
3. say why a GPT is **decoder-only**;
4. list the differences between the 2017 paper and our mini-GPT.

## 1. The original Transformer (Vaswani et al., 2017)

The Transformer was designed for **machine translation**: one sentence as input (English), one sentence as output (German). It has two halves:

<img src="https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer%2C_full_architecture.png" alt="Original encoder-decoder architecture of a Transformer" width="560" style="background:white"/>

*Illustration: dvgodoy, [Wikimedia Commons — Transformer, full architecture](https://commons.wikimedia.org/wiki/File:Transformer,_full_architecture.png), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), reproduced without modification.*

**How to read the diagram:**

- **The encoder** (left) reads **the whole source sentence**. Each word can look at **all** the others, before and after it: attention is **bidirectional**, with no mask. It produces a representation of the sentence.
- **The decoder** (right) writes the translation **word by word**. It contains two attentions:
  1. a **masked self-attention**: each word of the translation can only look at the words **already written**;
  2. a **cross-attention**: the *queries* come from the decoder, the *keys* and *values* come from the **encoder**. This is where the translation "consults" the source sentence.
- Each block also contains a **FeedForward**, **residual connections** ("Add") and **LayerNorms** ("Norm"), and the whole thing is repeated `N = 6` times.

## 2. Encoder-decoder vs decoder-only

![Encoder-decoder vs GPT](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_2/figures/04_encoder_decoder_vs_gpt.png)

Our task is not to translate but to **continue a text**. There is **no source sentence** to encode: there is only the text written so far. So we remove:

- the whole **encoder**;
- the decoder's **cross-attention** (it would have nothing to consult).

What remains is a stack of "masked self-attention + FeedForward" blocks: this is a **decoder-only Transformer**, the architecture of GPT models.

| Family | Attention | Example | Typical task |
|---|---|---|---|
| **Encoder-only** | bidirectional (no mask) | BERT | classify, understand a whole text |
| **Encoder-decoder** | bidirectional encoder + masked decoder + cross-attention | original Transformer, T5 | translate, summarize |
| **Decoder-only** | masked (causal) | GPT-2, GPT-3, LLaMA, **our mini-GPT** | continue a text |

> **Going further:** in his video, Karpathy devotes a section to this distinction (chapter *"encoder vs. decoder vs. both (?) Transformers"*, near the end). The deck `slides_encoder_decoder.md` covers these points as slides.

**A remark that will be useful in notebook 05:** the only code difference between an encoder self-attention and a decoder self-attention is **one line**: the one that applies the causal mask.

## 3. What we will implement

![The mini-GPT](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_2/figures/04_decoder_only.png)

Read the diagram **from bottom to top**, following the tensor shapes:

| Step | Operation | Shape | Notebook |
|---|---|---|---|
| Input | character indices | `(B, T)` | 01, 03 |
| Embeddings | token **+** position | `(B, T, C)` | 02, 08 |
| Block × `n_layer` | LayerNorm → **masked multi-head attention** → `+` | `(B, T, C)` | 05, 07 |
|  | LayerNorm → **FeedForward** → `+` | `(B, T, C)` | 06, 07 |
| Output | final LayerNorm → `lm_head` (Linear `C → V`) | `(B, T, V)` | 08 |
| Loss | reshape + `F.cross_entropy` | scalar | 02, 03 |

**What we already have:** the input, the output and the loss are **identical to the bigram**. The Transformer only changes what happens **in the middle**: the way the logits are produced.

**Karpathy's guiding idea:** in each block, we alternate
- **communication** (attention): positions exchange information, but only in one direction: future → past (each position looks back);
- **computation** (FeedForward): each position processes on its own what it has received.

## 4. Where are the parameters?

Without going into the code (that will be notebook 08), let's instantiate the model from `minigpt.py` and look at where the trainable parameters are.

In [1]:
# On Google Colab: fetch the course repository (corpora + shared modules such as minigpt.py).
# Locally, this cell does nothing.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("minigpt.py"):
    if not os.path.exists("ai-masterclass"):
        !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
    %cd ai-masterclass/en/chapter_2

In [2]:
# Setup: imports and the chapter's shared module
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

import minigpt  # the file minigpt.py, next to this notebook
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", device)

V = 65            # Shakespeare vocabulary
model = minigpt.GPTLanguageModel(vocab_size=V, block_size=64, n_embd=64, n_head=4, n_layer=4)

total = minigpt.count_parameters(model)
print(f"{'component':38s} {'parameters':>10s}   share")
rows = [("token embedding (V × C)", model.token_embedding_table),
        ("position embedding (T × C)", model.position_embedding_table)]
for i, block in enumerate(model.blocks):
    rows += [(f"block {i}: multi-head attention", block.sa),
             (f"block {i}: FeedForward", block.ffwd),
             (f"block {i}: 2 LayerNorms", nn.ModuleList([block.ln1, block.ln2]))]
rows += [("final LayerNorm", model.ln_f), ("lm_head (C × V + V)", model.lm_head)]
for name, module in rows:
    n = minigpt.count_parameters(module)
    print(f"{name:38s} {n:10,}   {100 * n / total:4.1f} %")
print(f"{'TOTAL':38s} {total:10,}")

PyTorch 2.15.0+cu132 | device: cuda


component                              parameters   share
token embedding (V × C)                     4,160    2.0 %
position embedding (T × C)                  4,096    1.9 %
block 0: multi-head attention              16,448    7.8 %
block 0: FeedForward                       33,088   15.6 %
block 0: 2 LayerNorms                         256    0.1 %
block 1: multi-head attention              16,448    7.8 %
block 1: FeedForward                       33,088   15.6 %
block 1: 2 LayerNorms                         256    0.1 %
block 2: multi-head attention              16,448    7.8 %
block 2: FeedForward                       33,088   15.6 %
block 2: 2 LayerNorms                         256    0.1 %
block 3: multi-head attention              16,448    7.8 %
block 3: FeedForward                       33,088   15.6 %
block 3: 2 LayerNorms                         256    0.1 %
final LayerNorm                               128    0.1 %
lm_head (C × V + V)                         4,225    2.0 

**Observations:**
- Most of the parameters are in the **blocks**, and in each block the **FeedForward** weighs about **twice** as much as the attention (it widens to `4C`, notebook 06).
- The causal mask does not appear: it is **not** a parameter.
- ~200,000 parameters: this is tiny (GPT-2 small: 124 M; GPT-3: 175 B), but **the architecture is the same**.

## 5. Our mini-GPT is not an exact copy of the paper

| | Transformer (2017) | Our mini-GPT |
|---|---|---|
| Structure | encoder + decoder | **decoder only**, no cross-attention |
| Tokens | subwords (BPE, ~37,000) | **characters** (65 for Shakespeare) |
| Position | **fixed** sinusoids | **learned** position embedding (like GPT-2) |
| Normalization | *post-norm*: `LayerNorm(x + f(x))` | *pre-norm*: `x + f(LayerNorm(x))` (like GPT-2) |
| Sizes | `d_model = 512`, `h = 8`, `N = 6`, `d_ff = 2048` | `C = 64`, `h = 4`, `n_layer = 4`, `4C = 256` |
| Activation | ReLU | ReLU (GPT-2 uses GELU) |

These differences do not change the **principle**. *Pre-norm* has become the standard because it makes training deep networks more stable (Xiong et al., 2020); we will see why in notebook 07.

## Summary

- Original Transformer = **encoder** (bidirectional attention) + **decoder** (masked attention + cross-attention to the encoder).
- **GPT = decoder-only**: no source text → no encoder, no cross-attention.
- A block = **communication** (masked attention) then **computation** (FeedForward), with residuals and LayerNorm, repeated `n_layer` times.
- Input, output and loss are those of the bigram: only **the middle** changes.

**Check your understanding**
1. Why doesn't the encoder's self-attention need a mask?
2. In cross-attention, where does Q come from? And K, V?
3. Which parts of the original diagram disappear in a GPT?

**Next → 05:** the central part, **masked attention**.

### References
- Vaswani et al., [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), NeurIPS 2017 — figure 1 and §3.
- Andrej Karpathy, [*Let's build GPT: from scratch, in code, spelled out*](https://www.youtube.com/watch?v=kCc8FmEb1nY) (2023) and its code [ng-video-lecture](https://github.com/karpathy/ng-video-lecture).
- A. Radford et al., [*Improving Language Understanding by Generative Pre-Training*](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) (GPT-1, 2018) and [*Language Models are Unsupervised Multitask Learners*](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (GPT-2, 2019).
- J. Devlin et al., [*BERT*](https://arxiv.org/abs/1810.04805), 2018 — the archetypal encoder-only model.
- R. Xiong et al., [*On Layer Normalization in the Transformer Architecture*](https://arxiv.org/abs/2002.04745), ICML 2020 — pre-norm vs post-norm.
- J. Alammar, [*The Illustrated Transformer*](https://jalammar.github.io/illustrated-transformer/) — an excellent visual explanation of the encoder-decoder.
- A. Rush, [*The Annotated Transformer*](https://nlp.seas.harvard.edu/annotated-transformer/) — the paper line by line, with code.